In [1]:
from langgraph.graph import StateGraph , START , END
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

True

In [3]:
model = ChatGroq(
    model  ='llama-3.3-70b-versatile'
)

In [4]:
class JokeState(TypedDict):
    topic:str 
    joke:str
    explanation:str

In [5]:
def make_joke(state: JokeState):
    prompt = f"Create a joke on topic : {state['topic']}"
    response = model.invoke(prompt).content

    return {'joke':response}

In [6]:
def explain_joke(state: JokeState):
    prompt = f"briefly Expalin the joke : {state['joke']} \n having topic : {state['topic']}"
    response = model.invoke(prompt).content

    return {'explanation' : response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('make_joke', make_joke)
graph.add_node('explain_joke', explain_joke)

graph.add_edge(START, 'make_joke')
graph.add_edge('make_joke', 'explain_joke')
graph.add_edge('explain_joke', END)

checkpoint= InMemorySaver()

workflow = graph.compile(checkpointer=checkpoint)

In [8]:
config1 = {'configurable': {'thread_id':'1'}}
final_state = workflow.invoke({'topic':'political situation of pakistan'}, config=config1)

In [9]:
config2 = {'configurable': {'thread_id':'2'}}
final_state = workflow.invoke({'topic':'Public and military of pakistan'}, config=config2)

In [10]:
final_state

{'topic': 'Public and military of pakistan',
 'joke': 'Why did the Pakistani military officer bring a ladder to the public gathering?\n\nBecause he wanted to take their relationship to a "higher" level, but little did he know, the public was already "ranks" above him in humor! (get it?)',
 'explanation': 'The joke is a play on words. The Pakistani military officer brings a ladder to a public gathering, intending to "take their relationship to a higher level" (meaning to improve or elevate it). However, the punchline is that the public is "ranks above him in humor", which is a pun on the military term "ranks" (referring to levels of seniority), but also implying that the public has a higher level of humor or wit, outsmarting the officer. It\'s a lighthearted joke poking fun at the military and the public\'s sense of humor in Pakistan.'}

In [11]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'political situation of pakistan', 'joke': 'Why did the politician in Pakistan bring a ladder to the parliament?\n\nBecause he wanted to take his promises to a "higher" level, but ended up "stepping down" anyway! (get it? like the frequent changes in government)', 'explanation': 'The joke is a play on words, referencing the frequent changes in government in Pakistan. The politician brings a ladder to "take his promises to a higher level", implying improvement. However, the punchline "stepping down" has a double meaning: both literally (getting down from the ladder) and figuratively (resigning from office), which is a common occurrence in Pakistan\'s political landscape. The joke humorously highlights the country\'s history of frequent government changes and political instability.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18f1b5-2868-67a4-8002-073050989ccf'}}, metadata={'source': 'loop', 'step': 2, 'par

In [12]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Public and military of pakistan', 'joke': 'Why did the Pakistani military officer bring a ladder to the public gathering?\n\nBecause he wanted to take their relationship to a "higher" level, but little did he know, the public was already "ranks" above him in humor! (get it?)', 'explanation': 'The joke is a play on words. The Pakistani military officer brings a ladder to a public gathering, intending to "take their relationship to a higher level" (meaning to improve or elevate it). However, the punchline is that the public is "ranks above him in humor", which is a pun on the military term "ranks" (referring to levels of seniority), but also implying that the public has a higher level of humor or wit, outsmarting the officer. It\'s a lighthearted joke poking fun at the military and the public\'s sense of humor in Pakistan.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f18f1b5-2f8d-6718-8002-f821a49e0722'}},

In [13]:
workflow.get_state({'configurable': {'thread_id':'2', 'checkpoint_id':'1f18f0c1-74ce-6d25-bfff-b61eed336d4f'}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f18f0c1-74ce-6d25-bfff-b61eed336d4f'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [14]:
workflow.invoke(None, {'configurable': {'thread_id':'2', 'checkpoint_id':'1f18f0c1-74ce-6d25-bfff-b61eed336d4f'}})

EmptyInputError: Received no input for __start__

In [15]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'political situation of pakistan', 'joke': 'Why did the politician in Pakistan bring a ladder to the parliament?\n\nBecause he wanted to take his promises to a "higher" level, but ended up "stepping down" anyway! (get it? like the frequent changes in government)', 'explanation': 'The joke is a play on words, referencing the frequent changes in government in Pakistan. The politician brings a ladder to "take his promises to a higher level", implying improvement. However, the punchline "stepping down" has a double meaning: both literally (getting down from the ladder) and figuratively (resigning from office), which is a common occurrence in Pakistan\'s political landscape. The joke humorously highlights the country\'s history of frequent government changes and political instability.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18f1b5-2868-67a4-8002-073050989ccf'}}, metadata={'source': 'loop', 'step': 2, 'pa

# updating state

In [16]:
workflow.update_state({'configurable': {'thread_id':'2', 'checkpoint_id':'1f18f090-dbe6-6b80-bfff-b8d7ffaf50f0'}}, {'topic':'ai in pakistan'})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f18f1e6-9953-6670-8000-0467591c75e7'}}

In [17]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'ai in pakistan'}, next=('make_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f18f1e6-9953-6670-8000-0467591c75e7'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-08-03T09:33:40.836332+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f18f090-dbe6-6b80-bfff-b8d7ffaf50f0'}}, tasks=(PregelTask(id='07f146f0-016c-9078-82f7-a352c98cb95d', name='make_joke', path=('__pregel_pull', 'make_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'Public and military of pakistan', 'joke': 'Why did the Pakistani military officer bring a ladder to the public gathering?\n\nBecause he wanted to take their relationship to a "higher" level, but little did he know, the public was already "ranks" above him in humor! (get it?)', 'explanation': 'The joke is a play on words. The Pakistani military offic

In [18]:
workflow.invoke(None, {'configurable': {'thread_id': '2', 'checkpoint_id': '1f18f0e7-077f-61e5-8000-9640d3467563'}})

EmptyInputError: Received no input for __start__

In [19]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'ai in pakistan'}, next=('make_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f18f1e6-9953-6670-8000-0467591c75e7'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-08-03T09:33:40.836332+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f18f090-dbe6-6b80-bfff-b8d7ffaf50f0'}}, tasks=(PregelTask(id='07f146f0-016c-9078-82f7-a352c98cb95d', name='make_joke', path=('__pregel_pull', 'make_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'Public and military of pakistan', 'joke': 'Why did the Pakistani military officer bring a ladder to the public gathering?\n\nBecause he wanted to take their relationship to a "higher" level, but little did he know, the public was already "ranks" above him in humor! (get it?)', 'explanation': 'The joke is a play on words. The Pakistani military offic

# Fault tolerance

In [20]:
class crashtype(TypedDict):
    input:str
    step1:str
    step2:str

In [21]:
import  time 
def step1(state: crashtype):
    print('Step 1')
    return {'step1':'done', 'input':state['input']}

def step2(state: crashtype):
    print('Step 2')
    time.sleep(1000)
    return {'step2':'done', 'input':state['input']}

def step3(state: crashtype):
    print('Step 3')
    return {'done':'True'}

In [22]:
graph_fault = StateGraph(crashtype)

graph_fault.add_node('step1', step1)
graph_fault.add_node('step2', step2)
graph_fault.add_node('step3', step3)

graph_fault.add_edge(START, 'step1')
graph_fault.add_edge('step1', 'step2')
graph_fault.add_edge('step2', 'step3')
graph_fault.add_edge('step3', END)

checkpointer = InMemorySaver()

workflow_fault = graph_fault.compile(checkpointer=checkpoint)

In [23]:
try:
    print('press any keyboard button to interrupt during step2')
    workflow_fault.invoke({'input':'start'}, config={'configurable': {'thread_id':'1'}})
except KeyboardInterrupt:
    print('Interrupt occur')

press any keyboard button to interrupt during step2
Step 1
Step 2
Interrupt occur


In [ ]:
print('rerunning the workflow after crash')
workflow_fault.invoke(None, config={'configurable': {'thread_id':'1'}})
print('workflow completed successfully after crash')

rerunning the workflow after crash
Step 2


In [ ]:
list(workflow_fault.get_state_history({'configurable': {'thread_id':'1'}}))